# Gradient Boosting

AdaBoost, Decision Trees, and Random Forests are prerequisites for Gradient Boosting.

## GB For Regression

We start by computing the average of all the numeric outputs (this is our initial guess). Of course, this average weight is very inaccurate, but it gives us a baseline.

Then, we compute the residuals between the actual and the average. This is a vector of all the errors:

$$
y' = y - \bar{y}
$$

Now, we try to construct a new tree that doesn't predict the actual weights, but instead the residuals. This tree is larger than a stump (like in AdaBoost), but still small (usually around 8 leaves). We construct the tree in the same way as we'd construct a regular decision tree, picking the best splits based on the Sum of Squared Residuals (SSR) error. If we call this tree the function $T_1$, then our updated prediction for each example $i$ is:

$$
y_{i, 1} = \bar{y} + T_1(i)
$$

However, this equation is incredibly prone to overfitting (low bias but high variance). So, we multiply the tree's output by a control called the __learning rate__ (value between 0 and 1). Smaller learning rate takes more trees/steps to get accurate results, but it prevents overfitting and leads to overall better predictions.

We can continuously create new trees that patch the remaining residuals that have not been handled by all the previously created trees.

### Mathematical Description

The input for Gradient Boosting is the input data $\{(x_i, y_i)\}_{i=1}^n$ and a differentiable Loss Function $L(y_i, F(X))$. For regression, this loss function is 1/2 of the SSR:
$$
L = \frac{1}{2} \sum_{i=1}^{n}(y_i - F(x_i))^2
$$

In our first step, we initialize the model with a constant value:
$$
F_0 (x) = \argmin_\gamma \sum_{i=1}^n L(y_i, \gamma)
$$

It turns out (pretty intuitively), that the constant $\gamma$ that minimizes the total loss is the average of the observed weights. Thus, $$F_0(x) = \bar{y}$$

In step 2, we make $M$ trees. It is standard for $M = 100$, meaning that we will construct 100 total trees to make the prediction. Here is how each of the $m \in [1, M]$ trees are constructed. First, we compute:

$$
r_{i, m} = - \left[ \frac{\partial L(y_i, F(x_i))}{\partial F(x_i)} \right]_{F(x) = F_{m-1}(x)} \forall i=1, \ldots, n
$$

For regression, this complex equation reduces quite nicely:

$$
\frac{\partial \frac{1}{2} \sum_{i=1}^{n}(y_i - F(x_i))^2}{\partial F(x_i)} = \frac{\partial \frac{1}{2} (y_i - F(x_i))^2}{\partial F(x_i)} = -(y_i - F(x_i)). 
$$

$$
r_{i, m} = - (-(y_i - F(x_i))) = y_i - F(x_i)
$$

Thus, $r_{i, m}$ is just the difference between the actual value and the current prediction. When $m=1$, this is just the actual minus the average of the observed ($F_0(x) = \bar{y}$). Each $r_{i, m}$ is called a __pseudo-residual__ because it is related to the idea of the residual between the observed and the predicted. 

The second step is to fit a regression tree to the $r_{i,m}$ values. Thus, we are predicting the residuals instead of the weights. We create terminal regions $R_{j, m}$ for $j = 1 \ldots J_m$, which for decision trees is just the leaves.

For each leaf, we need to compute the actual output value. In other words, for $j = 1 \ldots J_m$, we compute $\gamma_{j,m}$ such that:

$$
\gamma_{j,m} = \argmin_{\gamma} \sum_{x_i \in {R_{ij}}} L(y_i, F_{m-1}(x_i) + \gamma)
$$

With plugging in $L$ and some manipulation, it is not hard to see that the output for each leaf is optimized when it is simply the average of all the residuals for $x_i \in {R_{ij}}$.

Finally, we update the prediction with our information from the $mth$ tree. Our update is:

$$
F_m(x) = F_{m-1}(x) + \nu \sum_{j=1}^{J_m} \gamma_{jm} I(x \in R_{jm})
$$

At the end, gradient boost outputs $$F_M(x)$$

$F_M$ is the output after all $M$ trees have been computed. 

## GB for Classification

Like with regression, we start with a rough prediction. We can compute the log(odds) of all the data to create an initial probability prediction. We can use the logistic function:

$$
P = \frac{e^{\log(odds)}}{1 + e^{\log(odds)}}
$$

We can compute the residuals by calculating the difference between the predicted probability of a category and the actual category. For example, if we report a 0.7 probability of something being true, and the correct output is true, then we compute $1 - 0.7 = 0.3$.

We construct a tree to predict these residuals, and then the log-odds score we compute for each leaf is a transformation function from the probability residuals that will be discussed soon.